<div style="text-align:center; padding:20px 0">
<img src="https://raw.githubusercontent.com/dataprojectlabs/DataProjectLab-projects/refs/heads/main/media/logo_dataprojectlab.png" width="220"/>
</div>

# Notebook 4 — Dashboard Power BI & Storytelling
**SOLUTION**

**Projet :** EduTrack Analytics  
**Auteur :** DataProjectLab


---
## Architecture du dashboard EduTrack — 5 pages

| Page | Titre | Visuels cles |
|---|---|---|
| 1 | Vue executive | 4 KPIs + courbe inscriptions + anneau domaines + Top 5 parcours |
| 2 | Apprenants | Carte pays + pyramide ages + canaux acquisition + premium vs standard |
| 3 | Parcours & Instructeurs | Matrice performance + scatter CSAT x completion + heatmap activite |
| 4 | Revenus & Paiements | Courbe revenus + methodes paiement + panier moyen + top 5 parcours revenu |
| 5 | Alertes ML | Bandeau rouge + tableau at_risk + score_risque + filtre parcours |


---
## Etape 1 — Imports et verification des fichiers

### METHODE
Avant de charger dans Power BI, on verifie les types et les valeurs uniques des colonnes categoriques. Power BI est sensible a la casse et aux accents — `'Abandonne'` et `'Abandonné'` seraient traites comme deux valeurs differentes, cassant tous les filtres et mesures DAX bases sur ce statut.

In [1]:
import warnings
warnings.filterwarnings("ignore")
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", "{:.2f}".format)
COLORS = {"primary":"#534AB7","secondary":"#1D9E75","warning":"#EF9F27",
          "danger":"#E24B4A","neutral":"#888780","light":"#EEEDFE"}
plt.rcParams.update({"figure.facecolor":"white","axes.facecolor":"#F9F9F8",
                     "axes.grid":True,"grid.alpha":0.35,"font.size":11})
print("Configuration chargee ✓")

Configuration chargee ✓


In [2]:
df_parc  = pd.read_csv("../../dataset/parcours.csv")
df_app   = pd.read_csv("../../dataset/apprenants.csv",   parse_dates=["date_inscription"])
df_inscr = pd.read_csv("../../dataset/inscriptions.csv", parse_dates=["date_inscription","date_fin_reelle"])
df_sess  = pd.read_csv("../../dataset/sessions.csv",     parse_dates=["date_session"])
df_paie  = pd.read_csv("../../dataset/paiements.csv",    parse_dates=["date_paiement"])
df_an    = pd.read_csv("../../dataset/inscriptions_analytics.csv", parse_dates=["date_inscription"])
print("Chargement OK ✓")

Chargement OK ✓


In [3]:
# Verification des valeurs categoriques pour Power BI
print("=== VALEURS UNIQUES A VERIFIER ===")
print("statut :", df_an["statut"].unique())
print("methode:", df_paie["methode"].unique())
print("domaine:", df_parc["domaine"].unique())
print("pays (top 5):", df_app["pays"].value_counts().head(5).to_dict())

# Verification des types dates
print("\n=== TYPES DATES ===")
print(f"date_inscription : {df_an['date_inscription'].dtype}")
print(f"date_paiement    : {df_paie['date_paiement'].dtype}")

=== VALEURS UNIQUES A VERIFIER ===
statut : ['Termine' 'En cours' 'Abandonne' 'Suspendu']
methode: ['Carte bancaire' 'PayPal' 'Virement' 'Mobile Money']
domaine: ['Data & IA' 'Developpement Web' 'Marketing Digital' 'Management']
pays (top 5): {"Cote d'Ivoire": 1812, 'Senegal': 908, 'Cameroun': 700, 'Mali': 339, 'Burkina Faso': 321}

=== TYPES DATES ===
date_inscription : datetime64[ns]
date_paiement    : datetime64[ns]


### INTERPRETATION
Les valeurs categoriques sont propres et sans accents problematiques. Les colonnes de dates sont bien en `datetime64`. Tout est pret pour le chargement dans Power BI sans transformation supplementaire.

### METIER
La verification pre-Power BI evite des heures de debugging sur des mesures DAX qui retournent 0 a cause d'une difference de casse dans un filtre. C'est une etape systematique a ne jamais sauter.

---
## Etape 2 — Schema en etoile et relations

### METHODE
Le schema en etoile est le modele optimal pour Power BI. La table de faits centrale (`inscriptions_analytics`) contient les mesures et les cles etrangeres. Les dimensions (`parcours`, `apprenants`) contiennent les attributs descriptifs. **Regle d'or :** les relations vont toujours de la dimension vers le fait (direction du filtre = dimension → fait).

In [4]:
schema = """
SCHEMA EN ETOILE — EDUTRACK

      parcours (dimension)          Calendrier (dimension)
          |                              |
          | parcours_id                  | date
          |                              |
          +-------- inscriptions_analytics (FAIT) --------+
                    |                                      |
                    | apprenant_id              parcours_id|
                    |                                      |
               apprenants (dim)                  paiements
                                              (fait secondaire)
                                                   |
                                              apprenants_risque_scores
                                              (table alertes ML)

RELATIONS A CREER DANS POWER BI :
  parcours[parcours_id]  --> inscriptions_analytics[parcours_id]   (1:N, Single)
  apprenants[apprenant_id] --> inscriptions_analytics[apprenant_id] (1:N, Single)
  Calendrier[Date]       --> inscriptions_analytics[date_inscription] (1:N, Single)
  inscriptions_analytics[inscription_id] --> paiements[inscription_id] (1:N si disponible)
"""
print(schema)


SCHEMA EN ETOILE — EDUTRACK

      parcours (dimension)          Calendrier (dimension)
          |                              |
          | parcours_id                  | date
          |                              |
          +-------- inscriptions_analytics (FAIT) --------+
                    |                                      |
                    | apprenant_id              parcours_id|
                    |                                      |
               apprenants (dim)                  paiements
                                              (fait secondaire)
                                                   |
                                              apprenants_risque_scores
                                              (table alertes ML)

RELATIONS A CREER DANS POWER BI :
  parcours[parcours_id]  --> inscriptions_analytics[parcours_id]   (1:N, Single)
  apprenants[apprenant_id] --> inscriptions_analytics[apprenant_id] (1:N, Single)
  Calendrier[Date]      

---
## Etape 3 — Mesures DAX — Table _Mesures

### METHODE
Toutes les mesures sont creees dans une table vide `_Mesures` (table calculee). Cette convention permet de les retrouver facilement dans le panneau Champs. On utilise `VAR ... RETURN` pour toutes les mesures complexes — cela ameliore la lisibilite et les performances en evitant les recalculs.

In [1]:
# Mesures DAX a copier dans Power BI Desktop
DAX = {}

DAX["Total Inscriptions"] = """
Total Inscriptions = COUNTROWS(inscriptions_analytics)
"""

DAX["Taux Completion"] = """
Taux Completion =
DIVIDE(
    CALCULATE(COUNTROWS(inscriptions_analytics),
        inscriptions_analytics[statut] = "Termine"),
    [Total Inscriptions]
)
"""

DAX["Taux Abandon"] = """
Taux Abandon =
DIVIDE(
    CALCULATE(COUNTROWS(inscriptions_analytics),
        inscriptions_analytics[statut] = "Abandonne"),
    [Total Inscriptions]
)
"""

DAX["CSAT Moyen"] = """
CSAT Moyen =
CALCULATE(
    AVERAGE(inscriptions_analytics[csat]),
    inscriptions_analytics[statut] = "Termine"
)
"""

DAX["Revenu Total"] = """
Revenu Total =
CALCULATE(
    SUM(paiements[montant_fcfa]),
    paiements[statut_paiement] = "Valide"
)
"""

DAX["Nb Apprenants Risque"] = """
Nb Apprenants Risque =
VAR _result =
    CALCULATE(
        COUNTROWS(inscriptions_analytics),
        FILTER(
            inscriptions_analytics,
            inscriptions_analytics[at_risk_dropout] = 1
        )
    )
RETURN IF(ISBLANK(_result), 0, _result)
"""

DAX["Taux Risque"] = """
Taux Risque = DIVIDE([Nb Apprenants Risque], [Total Inscriptions])
"""

DAX["Engagement Moyen"] = """
Engagement Moyen = AVERAGE(inscriptions_analytics[engagement_score])
"""

DAX["Inscriptions Mois Prec"] = """
Inscriptions Mois Prec =
CALCULATE(
    [Total Inscriptions],
    PREVIOUSMONTH(Calendrier[Date])
)
"""

DAX["Variation Inscriptions"] = """
Variation Inscriptions =
DIVIDE(
    [Total Inscriptions] - [Inscriptions Mois Prec],
    [Inscriptions Mois Prec]
)
"""

DAX["Panier Moyen"] = """
Panier Moyen =
DIVIDE(
    [Revenu Total],
    CALCULATE(COUNTROWS(paiements),
        paiements[statut_paiement] = "Valide")
)
"""

DAX["Quadrant Instructeur"] = """
Quadrant Instructeur =
VAR csat  = [CSAT Moyen]
VAR compl = [Taux Completion]
VAR med_csat  = 4.24
VAR med_compl = 0.333
RETURN
SWITCH(TRUE(),
    csat >= med_csat && compl >= med_compl, "Top Performer",
    csat >= med_csat && compl <  med_compl, "Bon CSAT, Completion a ameliorer",
    csat <  med_csat && compl >= med_compl, "Bonne Completion, CSAT a ameliorer",
    "A accompagner"
)
"""

DAX["Nb Certificats"] = """
Nb Certificats =
CALCULATE(
    COUNTROWS(inscriptions_analytics),
    inscriptions_analytics[certificat_obtenu] = 1
)
"""

DAX["Sous Titre Page 1"] = """
Sous Titre Page 1 =
VAR _annee   = SELECTEDVALUE(Calendrier[Annee], "Toutes annees")
VAR _domaine = SELECTEDVALUE(parcours[domaine], "Tous domaines")
RETURN "EduTrack  ·  " & _annee & "  ·  " & _domaine
"""

print(f"Total mesures DAX documentees : {len(DAX)}")
for nom in DAX:
    print(f"  [{nom}]")

Total mesures DAX documentees : 14
  [Total Inscriptions]
  [Taux Completion]
  [Taux Abandon]
  [CSAT Moyen]
  [Revenu Total]
  [Nb Apprenants Risque]
  [Taux Risque]
  [Engagement Moyen]
  [Inscriptions Mois Prec]
  [Variation Inscriptions]
  [Panier Moyen]
  [Quadrant Instructeur]
  [Nb Certificats]
  [Sous Titre Page 1]


### INTERPRETATION
**14 mesures DAX** couvrent tous les KPIs du dashboard. Points cles a noter :

- `[Nb Apprenants Risque]` utilise `FILTER` (pas une comparaison directe dans CALCULATE) car `at_risk_dropout` est calcule, pas une colonne simple
- `[Inscriptions Mois Prec]` requiert que la table `Calendrier` soit **marquee comme table de dates** (`MARKASTABLE` dans Power BI) — sinon `PREVIOUSMONTH` ne fonctionne pas
- `[Quadrant Instructeur]` utilise les medianes reelles (4.24 et 33.3%) calculees dans le NB3

### METIER
La mesure `[Sous Titre Page 1]` rend le dashboard dynamique : quand le Dr. Konan filtre sur 'Data & IA' et '2023', le sous-titre affiche automatiquement 'EduTrack  ·  2023  ·  Data & IA'. C'est un detail UX qui profesionnalise le rendu.

---
## Etape 4 — Checklist de validation

Avant de livrer le dashboard, valider chaque KPI en comparant avec les valeurs du NB3 :

| Mesure DAX | Valeur attendue | Statut |
|---|---|---|
| [Total Inscriptions] (sans filtre) | 6 460 | ☐ |
| [Taux Completion] (sans filtre) | 33.3% | ☐ |
| [Taux Abandon] (sans filtre) | 17.7% | ☐ |
| [CSAT Moyen] (sans filtre) | 4.23 | ☐ |
| [Revenu Total] (sans filtre) | 748.7 M FCFA | ☐ |
| [Nb Apprenants Risque] (sans filtre) | 3 092 | ☐ |
| [Nb Certificats] (sans filtre) | 2 053 | ☐ |
| [Engagement Moyen] (sans filtre) | 7.88 | ☐ |

**Si un KPI ne correspond pas :** verifier les relations dans le modele de donnees, puis la direction du filtre (Single vs Bidirectional).

---

**DataProjectLab** — apprendre la data sur des cas concrets, structurés et orientés métier.